In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "utils"))

from animals_utils import SUBLIMINAL_PROMPT_TEMPLATES, RELATION_MAP, SYNONYM_ANIMALS, agroups, get_animals_of_groups, get_numbers
from data_loading import (
    load_all_logprobs, make_combination, get_logprob_diff,
    prepare_relation_data, get_common_animals,
    compute_cosine_similarities, load_dataset_frequency_ratios,
    DEFAULT_RESPONSE_START, RELATION_ORDER,
    )
from plotting import (
    scatter_logprob_vs_logprob, scatter_logprob_vs_logprob_mpl, scatter_logprob_vs_logprob_export, scatter_single_animal,
    scatter_animal_vs_animal, scatter_logprob_vs_metric,
    heatmap_correlations_by_animal, heatmap_correlations_by_relation,
    heatmap_correlations_by_animal_export, heatmap_correlations_by_relation_export,
    plot_similarity_heatmap, get_animal_color_map,
    )

results_dir = Path.cwd().parent / "results" / "Qwen2.5-7B-Instruct"
base_logprobs, subliminal_logprobs, unembedding_df = load_all_logprobs(results_dir)


animals = get_animals_of_groups([agroups.default])
#animals = get_animals_of_groups([agroups.single_token])
print("Animals found:", animals)

# Does loving a number influence other emotions towards animals?

In [ ]:
diff = lambda c: get_logprob_diff(c, subliminal_logprobs, base_logprobs)


love_love = make_combination("withoutthinking", "love", "love")
love_adore = make_combination("withoutthinking", "love", "adore")
love_like = make_combination("withoutthinking", "love", "like")
love_hate = make_combination("withoutthinking", "love", "hate")
love_detest = make_combination("withoutthinking", "love", "detest")
love_dislike = make_combination("withoutthinking", "love", "dislike")
hate_hate = make_combination("withoutthinking", "hate", "hate", baseline="allow_hate")

figures_dir = Path.cwd().parent / "report" / "figures"

scatter_logprob_vs_logprob_mpl(diff(love_love), diff(love_like), love_love, love_like, animals,
    note="Both axis are conditioned to love the numbers.<br>" \
    "X-axis is loving animals.<br>" \
    "Y-axis is liking animals.<br>" \
    "We expect a strong positive correlation here. Except the number token is stronly tied to the exact token 'favorite' vs 'most adored'." \
    "The strong and clear positive correlation shows that the same numbers are influencing both love and like.")

scatter_logprob_vs_logprob_mpl(diff(love_love), diff(love_hate), love_love, love_hate, animals,
    note="Both axis are conditioned to love the numbers.<br>" \
    "X-axis is loving animals.<br>" \
    "Y-axis is hating animals.<br>" \
    "We expect a strong negative correlation here. Since a loving a number that makes the llm love an animal, should also make it hate the same animal less." \
    "The missing correlation suggests that loving and hating an animal ist influenced by different numbers, even though same numbers influence love and like.")

scatter_logprob_vs_logprob_mpl(diff(love_hate), diff(love_dislike), love_hate, love_dislike, animals,
    note="Both axis are conditioned to love the numbers.<br>" \
    "X-axis is hating animals.<br>" \
    "Y-axis is disliking animals.<br>" \
    "We check whether negative emotions are surpressed in general or just steered by other numbers." \
    "The strong positive correlation shows, that the same numbers that steer hating an animal, also steer disliking it. This suggests that negative emotions are not suppressed in general, but just steered by different numbers than positive emotions.")

scatter_logprob_vs_logprob_export(
    diff(love_love),
    diff(love_like),
    love_love,
    love_like,
    x_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ love, animal)",
    y_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ like, animal)",
    animals=animals,
    export_path=figures_dir / "positive_negative_love_vs_like.png",
)

scatter_logprob_vs_logprob_export(
    diff(love_love),
    diff(love_hate),
    love_love,
    love_hate,
    x_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ love, animal)",
    y_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ hate, animal)",
    animals=animals,
    export_path=figures_dir / "positive_negative_love_vs_hate.png",
)

scatter_logprob_vs_logprob_export(
    diff(love_hate),
    diff(love_dislike),
    love_hate,
    love_dislike,
    x_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ hate, animal)",
    y_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ dislike, animal)",
    animals=animals,
    export_path=figures_dir / "positive_negative_hate_vs_dislike.png",
)

In [ ]:
# Correlation heatmaps by animal
from animals_utils import SINGLE_TOKEN_RELATIONS_QWEN, SINGLE_TOKEN_ANIMALS_QWEN


all_relation_data = prepare_relation_data(
    template_type="withoutthinking",
    number_relation="love",
    subliminal_logprobs=subliminal_logprobs,
    base_logprobs=base_logprobs,
    baseline="default",
)
print("Available relations:", list(all_relation_data.keys()))

synonym_animal_names = {a[0] for a in SYNONYM_ANIMALS}

non_synonym_animals = [a for a in get_common_animals(all_relation_data) if a not in synonym_animal_names]
non_synonym_animals = [a for a in get_common_animals(all_relation_data) if a not in SINGLE_TOKEN_ANIMALS_QWEN]

heatmap_correlations_by_animal(
    all_relation_data,
    title="Correlation Matrix per Animal: Logprob Differences for Different Animal Relations\n(Conditioned: Love without thinking about a number)",
    relation_order=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=non_synonym_animals,
    )

heatmap_correlations_by_animal_export(
    all_relation_data,
    export_path=figures_dir / "heatmap_corr_by_animal_single_token_relations.png",
    title="Correlation Matrix per Animal: Logprob Differences for Different Animal Relations\n(Conditioned: Love without thinking about a number)",
    relation_order=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=non_synonym_animals,
    )

heatmap_correlations_by_animal(
    all_relation_data,
    title="Correlation Matrix per Animal: Logprob Differences for RELATION_ORDER Animal Relations\n(Conditioned: Love without thinking about a number)",
    relation_order=RELATION_ORDER,
    animals=non_synonym_animals,
)

heatmap_correlations_by_animal_export(
    all_relation_data,
    export_path=figures_dir / "heatmap_corr_by_animal_selected_emotions.png",
    title="Correlation Matrix per Animal: Logprob Differences for RELATION_ORDER Animal Relations\n(Conditioned: Love without thinking about a number)",
    relation_order=RELATION_ORDER,
    animals=non_synonym_animals,
)

In [ ]:
# Scatter: average relation-pair correlation (r) vs unembedding cosine similarity
from data_loading import build_relation_pair_correlation_cosine_df
from plotting import scatter_relation_pair_avg_r_vs_unembedding_cosine, scatter_relation_pair_avg_r_vs_unembedding_cosine_export
from animals_utils import SINGLE_TOKEN_RELATIONS_QWEN

pair_df = build_relation_pair_correlation_cosine_df(
    all_relation_data=all_relation_data,
    unembedding_df=unembedding_df,
    relations=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=non_synonym_animals,
 )

pairs_to_label = [("hate", "dislike"), ("like", "cherish"), ("love", "hate"), ("like", "scorn"), ("tolerate", "reject")]

display(pair_df)
scatter_relation_pair_avg_r_vs_unembedding_cosine(
    pair_df,
    x_axis_text="Average Pearson r across animals (relation pair)",
    y_axis_text="Cosine similarity (relation unembedding vectors)",
    label_pairs=pairs_to_label,
)
scatter_relation_pair_avg_r_vs_unembedding_cosine_export(
    pair_df,
    x_axis_text="Average Pearson r across animals",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    export_path=figures_dir / "relation_pair_avg_r_vs_unembedding_cosine.png",
    label_pairs=pairs_to_label,
)

In [ ]:
# Animal-level analysis: average animal-animal correlation vs unembedding cosine
from data_loading import build_animal_pair_correlation_cosine_df
from plotting import heatmap_correlations_by_relation, heatmap_correlations_by_relation_export, scatter_relation_pair_avg_r_vs_unembedding_cosine, scatter_relation_pair_avg_r_vs_unembedding_cosine_export
from animals_utils import SINGLE_TOKEN_RELATIONS_QWEN, SINGLE_TOKEN_ANIMALS_QWEN

single_token_animals = [
    animal for animal in SINGLE_TOKEN_ANIMALS_QWEN
    if animal in get_common_animals(all_relation_data)
]


selected_animals = ["rabbit", "bunny", "hare", "mosquito", "cockroach", "cat", "kitty", "dog", "canine","panda", "kangaroo", "koala", "lion"]
print("Single-token animals used:", single_token_animals)
print("selected animals used:", selected_animals)

animal_pair_df = build_animal_pair_correlation_cosine_df(
    all_relation_data=all_relation_data,
    unembedding_df=unembedding_df,
    relations=SINGLE_TOKEN_RELATIONS_QWEN,
    animals=single_token_animals,
)

pairs_to_label = [("mouse", "rat"), ("mouse", "rabbit"), ("dog", "cat"), ("bunny", "cougar"), ("hog", "cattle"), ("rabbit", "bunny")]

scatter_relation_pair_avg_r_vs_unembedding_cosine(
    animal_pair_df,
    x_axis_text="Average Pearson r across relations (animal pair)",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    label_pairs=pairs_to_label,
)

scatter_relation_pair_avg_r_vs_unembedding_cosine_export(
    animal_pair_df,
    x_axis_text="Average Pearson r across relations (animal pair)",
    y_axis_text="Cosine similarity\n(unembedding vectors)",
    export_path=figures_dir / "animal_pair_avg_r_vs_unembedding_cosine.png",
    label_pairs=pairs_to_label,
)

for (group_name , used_animals) in [("single_token", single_token_animals), ("selected", selected_animals)]:
    heatmap_correlations_by_relation_export(
        all_relation_data,
        export_path=figures_dir / f"heatmap_corr_by_relation_{group_name}.png",
        animals=used_animals,
        relation_order=SINGLE_TOKEN_RELATIONS_QWEN,
        title="Correlation Matrix per Relation: Logprob Differences for Different Animals\n(Conditioned: Love without thinking about a number)",
    )


# Ist die Emotion zur Zahl auf das Tier übertragbar?

In [ ]:
scatter_logprob_vs_logprob_mpl(diff(love_love), diff(hate_hate), love_love, hate_hate, animals,
    note="The axis are conditioned differently.<br>" \
    "X-axis is loving numbers and loving animals.<br>" \
    "Y-axis is hating numbers and hating animals.<br>" \
    "We check whether the same numbers that influence loving an animal (when loving the number) also influence hating it (when hating the number)." \
    "If the correlation is positive, this would suggest, that the numbers are generally entangled with the animals, and hating is just suppressed by alignment and the alignment is supressed by prompting to hate a number.<br>" \
    "If the correlation is not present, the entanglement happens between triplets of (number, animal, emotion).<br>")

scatter_logprob_vs_logprob_export(
        diff(love_love),
        diff(hate_hate),
        love_love,
        hate_hate,
        x_axis_text=r"$\Delta_n^\text{em}$($e_n =$ love, $e_a =$ love, animal)",
        y_axis_text=r"$\Delta_n^\text{em}$($e_n =$ hate, $e_a =$ hate, animal)",
        animals=animals,
        export_path=figures_dir / "cross_condition_love_love_vs_hate_hate.png",
    )

# Allow hate

In [ ]:
love_love_allowhate = make_combination("withoutthinkingallowhate", "love", "love")
love_hate_allowhate = make_combination("withoutthinkingallowhate", "love", "hate")
love_detest_allowhate = make_combination("withoutthinkingallowhate", "love", "detest")

scatter_logprob_vs_logprob_mpl(diff(love_love_allowhate), diff(love_hate_allowhate), love_love_allowhate, love_hate_allowhate, animals,
    note="Both axis are conditioned to love the numbers. HATE is allowed now<br>" \
    "X-axis is loving animals.<br>" \
    "Y-axis is hating animals.<br>" \
    "Allowing to hate, makes to influencing numbers more similar.<br>" \
    "TODO: interpretation")

scatter_logprob_vs_logprob_mpl(diff(love_hate_allowhate), diff(love_detest_allowhate), love_hate_allowhate, love_detest_allowhate, animals,
    note="Both axis are conditioned to love the numbers. HATE is allowed now<br>" \
    "X-axis is hating animals.<br>" \
    "Y-axis is detesting animals.<br>" \
    "TODO" \
    "TODO")

scatter_logprob_vs_logprob_mpl(diff(love_hate), diff(love_hate_allowhate), love_hate, love_hate_allowhate, animals,
    note="Both axis are conditioned to love the numbers.<br>" \
    "X-axis is hating animals.<br>" \
    "Y-axis is hating animals (with HATE allowed).<br>" \
    "TODO" \
    "TODO")